# Daily Practice — 2026-10-29 — ML Testing & MLOps: Building a Data Validation & Reproducibility Gate

**Dataset:** [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) —
20,640 real 1990 census records for California block groups (median income, house age, average
rooms/bedrooms, population, average occupancy, latitude/longitude, and the target `MedHouseVal`,
median house value in $100,000s). Loaded at build time via `sklearn.datasets.fetch_california_housing`.

**Problem statement.** You've just joined an ML platform team as the embedded QA engineer for a
house-price model. The team's training pipeline currently has no guardrails: any CSV that lands in
the `incoming/` folder gets fed straight into `model.fit()`. You've seen this pattern fail before —
a silent schema change, a sensor that starts reporting `-1` instead of `NaN`, a unit change nobody
announced — and by the time anyone notices, a bad model is already serving predictions. Your job is
to build the two guardrails this pipeline is missing before you let it touch a single new model
version:

1. **A data validation gate** (`DataValidator`) that runs *before* training and rejects a batch that
   fails schema, range, missingness, or duplication checks — the same kind of contract testing you'd
   write for an API payload, applied to a training set.
2. **A reproducibility test** that proves "same data in, same seed, same model config" produces
   bit-for-bit identical evaluation metrics across two independent runs — and that a *different* seed
   or a *silently corrupted* input batch does not.

**What you should produce:** a working `DataValidator` class, a `train_and_score` function, a
reproducibility test that asserts exact-match metrics for a repeated run, and a demonstration that
your validator actually catches a corrupted production batch *before* it reaches training. Finish
with a short written answer (see the write-up cell) on what these checks do and don't protect you
from.

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option("display.width", 120)


## Load the data and define the contract

`fetch_california_housing` returns a clean, real dataset — no missing values, no duplicates, no
out-of-range values. That's exactly why we need to define the **contract** (expected schema and
expected value ranges) *now*, from the real, trusted data, before we ever see a corrupted batch. In
production this is the equivalent of writing down your API's request schema from a known-good
example payload.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
TARGET = "MedHouseVal"
FEATURES = [c for c in df.columns if c != TARGET]

print(df.shape)
df.head()


In [ ]:
# The contract, derived from the real data. In a real pipeline this would live in a
# schema file (e.g. a JSON/YAML contract or a great_expectations suite) checked into
# version control, not hardcoded like this -- but the content is the same idea.
EXPECTED_SCHEMA = {
    "MedInc": "float64",
    "HouseAge": "float64",
    "AveRooms": "float64",
    "AveBedrms": "float64",
    "Population": "float64",
    "AveOccup": "float64",
    "Latitude": "float64",
    "Longitude": "float64",
    "MedHouseVal": "float64",
}

# Generous ranges around the real observed min/max, with a little headroom --
# not the exact min/max, so the contract doesn't break on every legitimate new sample.
EXPECTED_RANGES = {
    "MedInc": (0.0, 20.0),
    "HouseAge": (0.0, 60.0),
    "AveRooms": (0.0, 50.0),
    "AveBedrms": (0.0, 10.0),
    "Population": (0.0, 40000.0),
    "AveOccup": (0.0, 20.0),
    "Latitude": (32.0, 42.5),
    "Longitude": (-124.5, -114.0),
    "MedHouseVal": (0.0, 5.01),  # the real target is capped at 5.00001 by the data source
}

MAX_MISSING_FRACTION = 0.01  # reject a batch with >1% nulls in any column


## Your task

Fill in the TODOs below. Signatures and class/function names are given — keep them as-is, the
solution cells (and the corruption demo) assume these names exist.

### Part A — `DataValidator`

Implement a validator with one method per check, each returning a small report, plus a `run_all`
that aggregates them. Each individual check method should return a dict shaped like:

```python
{"check": "<name>", "passed": bool, "details": "<human-readable summary>"}
```

- `validate_schema(df)` — every column in `EXPECTED_SCHEMA` is present, no unexpected extra columns,
  and dtypes match (treat any numeric dtype as compatible with `"float64"` if you like, but say so in
  `details`).
- `validate_ranges(df)` — every value in every expected numeric column falls inside
  `EXPECTED_RANGES[col]` (inclusive). Report which column(s) and how many rows failed.
- `validate_missingness(df)` — no column exceeds `MAX_MISSING_FRACTION` nulls.
- `validate_duplicates(df)` — no fully duplicated rows.
- `run_all(df)` — runs all four checks and returns `(passed: bool, reports: list[dict])`, where
  `passed` is `True` only if every individual check passed.

### Part B — training and reproducibility

- `train_and_score(df, seed)` — splits `df` into train/test (use `seed` for both the split and the
  model), fits a `RandomForestRegressor`, and returns a dict of metrics, e.g.
  `{"mae": ..., "r2": ...}`.
- `test_reproducibility()` — calls `train_and_score(df, seed=7)` twice and asserts the two metric
  dicts are *exactly* equal. Then calls it once more with a different seed and shows (print, don't
  assert) that the metrics differ from the seed-7 run. Also compute and print a content hash of
  `df` (e.g. via `pandas.util.hash_pandas_object` or hashing `df.to_csv()`) alongside each run's
  metrics — this is what you'd log in a real experiment tracker so you can tell *which exact data*
  produced *which exact metrics*.

### Part C — catch a corrupted batch before it ever reaches training

- `corrupt_batch(df)` — return a corrupted **copy** of `df` that: (a) sets `AveRooms` to `-1` for a
  random 5% of rows (simulating a sensor/ETL bug that reports a sentinel value instead of `NaN`),
  and (b) duplicates a random 2% of rows onto the end (simulating an upstream join bug). Don't modify
  `df` itself.
- Run `DataValidator().run_all(...)` on both the clean `df` and on `corrupt_batch(df)`, and print
  both reports side by side. The clean batch should pass every check; the corrupted batch should fail
  at least the range check and the duplicate check — and your training pipeline should refuse to call
  `train_and_score` on it.

In [ ]:
class DataValidator:
    def validate_schema(self, df: pd.DataFrame) -> dict:
        # TODO
        raise NotImplementedError

    def validate_ranges(self, df: pd.DataFrame) -> dict:
        # TODO
        raise NotImplementedError

    def validate_missingness(self, df: pd.DataFrame) -> dict:
        # TODO
        raise NotImplementedError

    def validate_duplicates(self, df: pd.DataFrame) -> dict:
        # TODO
        raise NotImplementedError

    def run_all(self, df: pd.DataFrame):
        # TODO: call the four checks above and aggregate
        raise NotImplementedError


In [ ]:
def train_and_score(df: pd.DataFrame, seed: int) -> dict:
    # TODO: split df into train/test using `seed`, fit a RandomForestRegressor(random_state=seed),
    # return {"mae": ..., "r2": ...} on the test set
    raise NotImplementedError


def test_reproducibility():
    # TODO: run train_and_score(df, seed=7) twice, assert the metric dicts are exactly equal.
    # Run once more with a different seed and print how the metrics differ.
    # Print a content hash of df alongside each run's metrics.
    raise NotImplementedError


In [ ]:
def corrupt_batch(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: return a corrupted COPY of df (see Part C for the exact corruption to apply)
    raise NotImplementedError


# TODO: run DataValidator().run_all() on the clean df and on corrupt_batch(df), print both reports


### Write-up

TODO: in 3–5 sentences — your `EXPECTED_RANGES` were set by hand from one snapshot of real data.
What happens to this validation gate the first time a *legitimate* new batch includes, say, a block
group with `AveOccup` higher than anything in the training snapshot? Is a hand-set range check from
one dataset snapshot a reasonable permanent contract, or does it need a maintenance plan — and if so,
what would trigger an update? Separately: your reproducibility test only proves the *pipeline* is
deterministic given the *same* `df`. What does it *not* prove about reproducibility if `df` itself
was pulled fresh from a live source at each run?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
class DataValidator:
    def validate_schema(self, df: pd.DataFrame) -> dict:
        expected_cols = set(EXPECTED_SCHEMA)
        actual_cols = set(df.columns)
        missing = expected_cols - actual_cols
        extra = actual_cols - expected_cols
        dtype_mismatches = {
            c: str(df[c].dtype)
            for c in expected_cols & actual_cols
            if not np.issubdtype(df[c].dtype, np.floating)
        }
        passed = not missing and not extra and not dtype_mismatches
        details = (
            f"missing={sorted(missing)}, extra={sorted(extra)}, "
            f"dtype_mismatches={dtype_mismatches}"
        )
        return {"check": "schema", "passed": passed, "details": details}

    def validate_ranges(self, df: pd.DataFrame) -> dict:
        failures = {}
        for col, (lo, hi) in EXPECTED_RANGES.items():
            if col not in df.columns:
                continue
            bad = ~df[col].between(lo, hi)
            n_bad = int(bad.sum())
            if n_bad:
                failures[col] = n_bad
        passed = len(failures) == 0
        return {"check": "ranges", "passed": passed, "details": f"out-of-range counts: {failures}"}

    def validate_missingness(self, df: pd.DataFrame) -> dict:
        frac_null = df.isna().mean()
        offenders = frac_null[frac_null > MAX_MISSING_FRACTION]
        passed = offenders.empty
        return {
            "check": "missingness",
            "passed": passed,
            "details": f"columns over {MAX_MISSING_FRACTION:.0%} null: {offenders.to_dict()}",
        }

    def validate_duplicates(self, df: pd.DataFrame) -> dict:
        n_dupes = int(df.duplicated().sum())
        return {
            "check": "duplicates",
            "passed": n_dupes == 0,
            "details": f"{n_dupes} fully duplicated row(s)",
        }

    def run_all(self, df: pd.DataFrame):
        reports = [
            self.validate_schema(df),
            self.validate_ranges(df),
            self.validate_missingness(df),
            self.validate_duplicates(df),
        ]
        passed = all(r["passed"] for r in reports)
        return passed, reports


def train_and_score(df: pd.DataFrame, seed: int) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        df[FEATURES], df[TARGET], test_size=0.2, random_state=seed
    )
    model = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return {
        "mae": round(float(mean_absolute_error(y_test, preds)), 6),
        "r2": round(float(r2_score(y_test, preds)), 6),
    }


def hash_df(df: pd.DataFrame) -> str:
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:12]


def test_reproducibility():
    run_a = train_and_score(df, seed=7)
    run_b = train_and_score(df, seed=7)
    assert run_a == run_b, f"Same seed produced different metrics: {run_a} vs {run_b}"

    run_c = train_and_score(df, seed=99)
    print(f"df hash: {hash_df(df)}")
    print(f"seed=7  run A: {run_a}")
    print(f"seed=7  run B: {run_b}  (exact match with run A: {run_a == run_b})")
    print(f"seed=99 run C: {run_c}  (differs from seed=7, as expected)")


test_reproducibility()


def corrupt_batch(df: pd.DataFrame) -> pd.DataFrame:
    corrupted = df.copy()
    rng = np.random.RandomState(0)

    n_rows = len(corrupted)
    sentinel_idx = rng.choice(corrupted.index, size=int(0.05 * n_rows), replace=False)
    corrupted.loc[sentinel_idx, "AveRooms"] = -1

    dupe_idx = rng.choice(corrupted.index, size=int(0.02 * n_rows), replace=False)
    corrupted = pd.concat([corrupted, corrupted.loc[dupe_idx]], ignore_index=True)

    return corrupted


clean_passed, clean_reports = DataValidator().run_all(df)
dirty_passed, dirty_reports = DataValidator().run_all(corrupt_batch(df))

print(f"\nClean batch passed overall: {clean_passed}")
for r in clean_reports:
    print(" ", r)

print(f"\nCorrupted batch passed overall: {dirty_passed}")
for r in dirty_reports:
    print(" ", r)

if not dirty_passed:
    print("\nGate correctly blocked the corrupted batch -- train_and_score() was never called on it.")
```

**Why this works:** `validate_ranges` flags the sentinel `-1` values in `AveRooms` because `-1` falls
outside the real, physically-sensible range for "average rooms per household" — the same instinct an
SQA engineer uses to write a boundary-value test for an API field, just applied to a training column.
`validate_duplicates` catches the join-bug duplicates directly. Neither check needed to know *why*
the batch was corrupted, only what "valid" looks like — which is the point: a good data contract
catches classes of bugs, not specific bugs. The reproducibility test is deliberately strict
(`==`, not "close enough") because `RandomForestRegressor` with a fixed `random_state` and
scikit-learn's deterministic tree-building *should* give you bit-identical output on the same data;
if it ever doesn't, that's a real signal (an unseeded source of randomness, parallelism
nondeterminism, or a changed library version) worth investigating, not something to average away.

</details>